# Lakeflow Declarative Pipeline

This notebook implements a Medallion Architecture using Databricks Lakeflow Declarative Pipelines.

Layers:
Bronze
Silver
Gold

Features:
- Streaming Tables
- Materialized Views
- Data Quality Rules
- Incremental Processing

In [0]:
-- ==========================================================
-- Bronze Layer
-- Create Materialized View for Diagnosis Mapping
-- ==========================================================

CREATE OR REFRESH MATERIALIZED VIEW incremental_load.default.diagnostic_mapping_v1
COMMENT "Bronze table for the diagnosis mapping file"
AS
SELECT *
FROM incremental_load.default.raw_diagnosis_map;

In [0]:
 -- ==========================================================
-- Bronze Layer
-- Create Streaming Table for Patient Data
-- ==========================================================
 
 CREATE OR REFRESH STREAMING TABLE daily_patients
 COMMENT "Bronze table for daily patient data"
 TBLPROPERTIES("quality"= "bronze")
 AS
 SELECT *
 FROM STREAM(incremental_load.default.raw_patients_daily);


In [0]:
-- ==========================================================
-- Silver Layer
-- Join Patient Data with Diagnosis Mapping
-- Apply Data Quality Constraints
-- ==========================================================

CREATE OR REFRESH STREAMING TABLE processed_patient_data (
  CONSTRAINT valid_data EXPECT (
    patient_id IS NOT NULL AND 
    name IS NOT NULL AND 
    age IS NOT NULL AND 
    gender IS NOT NULL AND 
    address IS NOT NULL AND 
    contact_number IS NOT NULL AND 
    admission_date IS NOT NULL 
  ) ON VIOLATION DROP ROW
)
COMMENT "Silver table with newly joined data from bronze tables and data quality checks"
TBLPROPERTIES("quality"= "silver")
AS
SELECT
  p.patient_id,
  p.name,
  p.age,
  p.gender,
  p.address,
  p.contact_number,
  p.admission_date,
  m.diagnosis_description  
FROM
  STREAM(live.daily_patients) p
  LEFT JOIN live.diagnostic_mapping_v1 m
  ON p.diagnosis_code = m.diagnosis_code;

In [0]:
-- ==========================================================
-- Gold Layer
-- Analytics Tables (Daignosis Description)
-- ==========================================================

CREATE OR REFRESH MATERIALIZED VIEW incremental_load.default.patient_statistics_by_diagnosis
COMMENT "Gold table with detailed patient statistics by diagnosis"
TBLPROPERTIES("quality"= "gold")
AS
SELECT
    diagnosis_description,  
    COUNT(patient_id) AS patient_count,
    AVG(age) AS avg_age,
    COUNT(DISTINCT gender) AS unique_gender_count,
    MIN(age) as min_age,
    MAX(age) as max_age   
FROM incremental_load.default.processed_patient_data  
GROUP BY diagnosis_description;

In [0]:
-- ==========================================================
-- Gold Layer
-- Analytics Tables (Gender)
-- ==========================================================

CREATE OR REFRESH MATERIALIZED VIEW incremental_load.default.patient_statistics_by_gender
COMMENT "Gold table with detailed patient statistics by gender"
TBLPROPERTIES("quality"= "gold")
AS
SELECT
    gender,
    COUNT(patient_id) AS patient_count,
    AVG(age) AS avg_age,
    COUNT(DISTINCT diagnosis_description) AS unique_diagnosis_count,  
    MIN(age) as min_age,
    MAX(age) as max_age
FROM incremental_load.default.processed_patient_data 
GROUP BY gender;